# IMPORTING LIBRARIES

In [31]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import RFE
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
df = pd.read_csv('customer_churn.csv')

In [4]:
df.head()

,Customer_ID,Age,Tenure,ServicePlan,PaymentMethod,MonthlyUsage,SupportCalls,Churn
0,0,56,51,Premium,BankTransfer,NaN,0,0
1,1,69,13,Premium,Cash,82,0,1
2,2,46,57,Basic,BankTransfer,284,0,0
3,3,32,55,Standard,BankTransfer,84,3,1
4,4,60,28,Premium,CreditCard,94,1,0


In [5]:
df.info(), df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23053 entries, 0 to 23052
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Customer_ID    23053 non-null  int64 
 1   Age            23053 non-null  int64 
 2   Tenure         23053 non-null  int64 
 3   ServicePlan    23053 non-null  object
 4   PaymentMethod  23053 non-null  object
 5   MonthlyUsage   20747 non-null  object
 6   SupportCalls   23053 non-null  object
 7   Churn          23053 non-null  int64 
dtypes: int64(4), object(4)
memory usage: 1.4+ MB


(None,
 Customer_ID         0
 Age                 0
 Tenure              0
 ServicePlan         0
 PaymentMethod       0
 MonthlyUsage     2306
 SupportCalls        0
 Churn               0
 dtype: int64)

In [6]:
df['MonthlyUsage'] = pd.to_numeric(df['MonthlyUsage'] , errors='coerce')
df['SupportCalls'] = pd.to_numeric(df['SupportCalls'] , errors='coerce')


In [7]:
df.head()

,Customer_ID,Age,Tenure,ServicePlan,PaymentMethod,MonthlyUsage,SupportCalls,Churn
0,0,56,51,Premium,BankTransfer,NaN,0.0,0
1,1,69,13,Premium,Cash,82.0,0.0,1
2,2,46,57,Basic,BankTransfer,284.0,0.0,0
3,3,32,55,Standard,BankTransfer,84.0,3.0,1
4,4,60,28,Premium,CreditCard,94.0,1.0,0


In [8]:
df.duplicated().any().sum()

np.int64(0)

# MISSING VALUE IMPUTATION

In [9]:
imputer = KNNImputer(n_neighbors= 5)
 
cols_to_impute = ['Age', 'Tenure', 'MonthlyUsage', 'SupportCalls']

df[cols_to_impute] = imputer.fit_transform(df[cols_to_impute])

In [10]:
df.isna().sum()

Customer_ID      0
Age              0
Tenure           0
ServicePlan      0
PaymentMethod    0
MonthlyUsage     0
SupportCalls     0
Churn            0
dtype: int64

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23053 entries, 0 to 23052
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Customer_ID    23053 non-null  int64  
 1   Age            23053 non-null  float64
 2   Tenure         23053 non-null  float64
 3   ServicePlan    23053 non-null  object 
 4   PaymentMethod  23053 non-null  object 
 5   MonthlyUsage   23053 non-null  float64
 6   SupportCalls   23053 non-null  float64
 7   Churn          23053 non-null  int64  
dtypes: float64(4), int64(2), object(2)
memory usage: 1.4+ MB


# ONE HOT ENCODING

In [12]:
df['PaymentMethod'].unique() , df['ServicePlan'].unique()

(array(['BankTransfer', 'Cash', 'CreditCard'], dtype=object),
 array(['Premium', 'Basic', 'Standard'], dtype=object))

In [13]:
df = pd.get_dummies(df , columns=['ServicePlan' ,'PaymentMethod' ] , drop_first=True).astype(int)

In [14]:
df

,Customer_ID,Age,Tenure,MonthlyUsage,SupportCalls,Churn,ServicePlan_Premium,ServicePlan_Standard,PaymentMethod_Cash,PaymentMethod_CreditCard
0,0,56,51,175,0,0,1,0,0,0
1,1,69,13,82,0,1,1,0,1,0
2,2,46,57,284,0,0,0,0,0,0
3,3,32,55,84,3,1,0,1,0,0
4,4,60,28,94,1,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...
23048,23048,31,23,73,4,1,0,1,0,1
23049,23049,29,61,196,6,0,1,0,0,0
23050,23050,67,7,165,5,0,0,0,1,0
23051,23051,21,44,153,8,0,1,0,0,1


# SPLITING

In [15]:
x = df.drop(columns=['Customer_ID' ,'Churn' ])
y = df['Churn']

In [16]:
x_train , x_test , y_train , y_test = train_test_split(x , y , test_size=0.2 , random_state=30)

In [20]:
dt = DecisionTreeClassifier(random_state=42)

rfe = RFE(
    estimator=dt,
    n_features_to_select=5
)

rfe.fit(x_train,y_train)

selcted_feature = x_train.columns[rfe.support_]

print("Selected Features:", selcted_feature)

Selected Features: Index(['Age', 'Tenure', 'MonthlyUsage', 'SupportCalls', 'ServicePlan_Premium'], dtype='object')


In [22]:
x_train_fs = x_train[selcted_feature]
x_test_fs = x_test[selcted_feature]

In [23]:
param_grid = {
    'max_depth': [3,5,7,10,None],
    'min_samples_split': [2,5,10],
    'min_samples_leaf': [1,2,4],
    'criterion': ['gini','entropy','log_loss'],
    'max_features': [None,'sqrt','log2']
}

In [24]:
grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(x_train_fs, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy', 'log_loss'],
                         'max_depth': [3, 5, 7, 10, None],
                         'max_features': [None, 'sqrt', 'log2'],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10]},
             scoring='accuracy')

In [25]:
best_model = grid.best_estimator_

print("Best Parameters:", grid.best_params_)

Best Parameters: {'criterion': 'entropy', 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5}


In [26]:
y_pred = best_model.predict(x_test_fs)

In [27]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.5096508349598785

Classification Report:
               precision    recall  f1-score   support

           0       0.51      0.49      0.50      2319
           1       0.51      0.53      0.52      2292

    accuracy                           0.51      4611
   macro avg       0.51      0.51      0.51      4611
weighted avg       0.51      0.51      0.51      4611

